## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

pd.set_option('display.max_columns', None)


## 2. Load the Dataset

> **Note (Google Colab):** Upload `seasonal_agriculture_performance_dataset.csv` to the Colab session (folder icon on the left → Upload), or mount Google Drive, then adjust the path below if needed.


In [ ]:
df = pd.read_csv('seasonal_agriculture_performance_dataset.csv')
df.head()


In [ ]:
print("Shape:", df.shape)
df.info()


## 3. Initial Data Exploration


In [ ]:
df.describe().T


In [ ]:
print("Categorical columns and unique values:\n")
for col in ['State', 'District', 'Crop', 'Season', 'Irrigation_Method']:
    print(f"{col}: {df[col].nunique()} unique -> {df[col].unique() if df[col].nunique() < 10 else df[col].unique()[:10]}")


In [ ]:
print("Missing values per column:")
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing)
print(f"\nTotal missing cells: {missing.sum()} ({missing.sum() / df.size * 100:.2f}% of dataset)")


In [ ]:
# Check for duplicate Farm_IDs
print("Duplicate Farm_ID rows:", df['Farm_ID'].duplicated().sum())
print("Fully duplicate rows:", df.duplicated().sum())


## 4. Data Cleaning and Preparation

Missing values exist in `Rainfall_mm`, `Soil_Moisture_pct`, and `Yield_Tonnes_Ha` (all under 1.5% of rows). Since these are seasonal/environmental measurements, we impute them using the **median value for that Season + Crop combination**, which preserves seasonal and crop-specific patterns better than a single global median.


In [ ]:
df_clean = df.copy()

cols_to_impute = ['Rainfall_mm', 'Soil_Moisture_pct', 'Yield_Tonnes_Ha']

for col in cols_to_impute:
    df_clean[col] = df_clean.groupby(['Season', 'Crop'])[col].transform(
        lambda x: x.fillna(x.median())
    )

# Fallback: fill any remaining NaNs with overall median (in case a Season+Crop group was entirely NaN)
for col in cols_to_impute:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Remaining missing values:", df_clean.isnull().sum().sum())


**Note on Yield and Water Efficiency scale:** Sugarcane yields (30-100+ tonnes/hectare) are naturally an order of magnitude higher than other crops (0.3-6 t/ha) because sugarcane is a high-biomass crop. These are genuine values, not data errors, so per-crop or per-crop-normalized comparisons are used throughout rather than treating them as outliers to remove.


In [ ]:
# Derived metrics useful for seasonal comparison
df_clean['Profit_Margin_pct'] = (df_clean['Profit_INR'] / df_clean['Revenue_INR']) * 100
df_clean['Cost_per_Hectare'] = df_clean['Total_Cost_INR'] / df_clean['Farm_Area_Hectares']
df_clean['Is_Profitable'] = df_clean['Profit_INR'] > 0

df_clean[['Profit_Margin_pct', 'Cost_per_Hectare', 'Is_Profitable']].describe(include='all')


## 5. Seasonal Overview

How much data do we have per season, and how are crops distributed across seasons?


In [ ]:
season_counts = df_clean['Season'].value_counts()
print(season_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

season_counts.plot(kind='bar', ax=axes[0], color=['#4C956C', '#D68C45', '#3E7CB1'])
axes[0].set_title('Number of Farm Records per Season')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

crop_season = pd.crosstab(df_clean['Crop'], df_clean['Season'])
crop_season.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2')
axes[1].set_title('Crop Distribution Across Seasons')
axes[1].set_xlabel('Crop')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 6. Environmental Conditions Across Seasons

**Key question:** How do rainfall, temperature, humidity, and sunlight differ between Kharif, Rabi and Zaid?


In [ ]:
env_cols = ['Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct', 'Sunlight_Hours_Day', 'Soil_Moisture_pct']

season_env = df_clean.groupby('Season')[env_cols].mean().round(2)
season_env


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(env_cols):
    sns.boxplot(data=df_clean, x='Season', y=col, hue='Season', ax=axes[i], palette='Set2', legend=False,
                order=['Kharif', 'Rabi', 'Zaid'])
    axes[i].set_title(col.replace('_', ' '))

axes[-1].axis('off')
plt.suptitle('Environmental Conditions by Season', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


**Observation:** Kharif (monsoon season) shows the highest rainfall and humidity, consistent with India's monsoon-driven cropping calendar. Zaid (summer season) shows higher temperatures and lower rainfall, requiring more irrigation dependence. Rabi (winter season) sits in between with moderate rainfall and cooler temperatures.


## 7. Resource Usage Across Seasons

**Key question:** Are there noticeable variations in resource usage (water, fertilizer, pesticide) across seasons?


In [ ]:
resource_cols = ['Fertilizer_kg_ha', 'Pesticide_Litre_ha', 'Water_Used_m3', 'Nitrogen_kg_ha', 'Phosphorus_kg_ha', 'Potassium_kg_ha']

season_resources = df_clean.groupby('Season')[resource_cols].mean().round(2)
season_resources


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

season_resources[['Fertilizer_kg_ha', 'Pesticide_Litre_ha']].plot(kind='bar', ax=axes[0], colormap='viridis')
axes[0].set_title('Avg Fertilizer & Pesticide Use by Season')
axes[0].tick_params(axis='x', rotation=0)

sns.boxplot(data=df_clean, x='Season', y='Water_Used_m3', hue='Season', ax=axes[1], palette='Blues', legend=False,
            order=['Kharif', 'Rabi', 'Zaid'])
axes[1].set_title('Water Used (m³) by Season')

plt.tight_layout()
plt.show()


In [ ]:
# Irrigation method preference by season
irrigation_season = pd.crosstab(df_clean['Season'], df_clean['Irrigation_Method'], normalize='index') * 100
irrigation_season = irrigation_season.round(1)
print(irrigation_season)

irrigation_season.plot(kind='bar', stacked=True, figsize=(9, 5), colormap='coolwarm')
plt.title('Irrigation Method Share by Season (%)')
plt.ylabel('% of Farms')
plt.xticks(rotation=0)
plt.legend(title='Irrigation Method', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


**Observation:** Zaid season farms rely more heavily on drip/sprinkler irrigation given lower natural rainfall, while Kharif season shows more rainfed and flood irrigation, aligning with monsoon water availability.


## 8. Crop Yield Across Seasons

Since yield scale differs drastically by crop (sugarcane vs others), we compare **within each crop** across seasons.


In [ ]:
pivot_yield = df_clean.pivot_table(values='Yield_Tonnes_Ha', index='Crop', columns='Season', aggfunc='mean').round(2)
pivot_yield


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(pivot_yield, annot=True, fmt='.1f', cmap='YlGn', ax=ax, cbar_kws={'label': 'Avg Yield (t/ha)'})
ax.set_title('Average Yield by Crop and Season')
plt.tight_layout()
plt.show()


In [ ]:
# Normalized comparison: yield relative to each crop's own overall mean, to compare seasonal effect fairly
norm = df_clean.copy()
norm['Yield_Normalized'] = norm.groupby('Crop')['Yield_Tonnes_Ha'].transform(lambda x: x / x.mean())

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=norm, x='Season', y='Yield_Normalized', hue='Season', order=['Kharif', 'Rabi', 'Zaid'], palette='Set2', ax=ax, legend=False)
ax.axhline(1.0, color='red', linestyle='--', alpha=0.6, label='Crop average')
ax.set_title('Yield Relative to Each Crop\'s Own Average, by Season')
ax.set_ylabel('Yield / Crop Mean Yield')
ax.legend()
plt.tight_layout()
plt.show()


## 9. Economic Performance Across Seasons

**Key question:** How do economic outcomes (revenue, cost, profit) vary across seasons?


In [ ]:
econ_cols = ['Total_Cost_INR', 'Revenue_INR', 'Profit_INR', 'Profit_Margin_pct']
season_econ = df_clean.groupby('Season')[econ_cols].mean().round(0)
season_econ


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

season_econ[['Total_Cost_INR', 'Revenue_INR']].plot(kind='bar', ax=axes[0], color=['#C1440E', '#2E8B57'])
axes[0].set_title('Avg Cost vs Revenue by Season')
axes[0].set_ylabel('INR')
axes[0].tick_params(axis='x', rotation=0)

sns.boxplot(data=df_clean, x='Season', y='Profit_INR', hue='Season', order=['Kharif', 'Rabi', 'Zaid'], palette='Set2', ax=axes[1], legend=False)
axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[1].set_title('Profit Distribution by Season')

plt.tight_layout()
plt.show()


In [ ]:
profitability = df_clean.groupby('Season')['Is_Profitable'].mean().mul(100).round(1)
print("% of farms that were profitable, by season:")
print(profitability)

profitability.plot(kind='bar', color=['#4C956C', '#D68C45', '#3E7CB1'], figsize=(7, 5))
plt.title('Share of Profitable Farms by Season')
plt.ylabel('% Profitable')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Observation:** Note that a large share of farms in this dataset run at a loss overall (Total_Cost often exceeds Revenue). This pattern is worth calling out explicitly in the report as a key finding, and it's useful to check whether unprofitability concentrates in specific seasons, crops, or regions (explored next).


## 10. Water Use Efficiency Across Seasons

**Key question:** How efficiently is water being used relative to yield, and does this vary by season?


In [ ]:
pivot_we = df_clean.pivot_table(values='Water_Efficiency_t_per_1000m3', index='Crop', columns='Season', aggfunc='mean').round(2)
pivot_we


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(pivot_we, annot=True, fmt='.1f', cmap='Blues', ax=ax, cbar_kws={'label': 'Tonnes per 1000 m³'})
ax.set_title('Water Use Efficiency by Crop and Season')
plt.tight_layout()
plt.show()


## 11. Disease & Pest Risk Across Seasons

**Key question:** Are disease/pest risk patterns consistent across seasons?


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_clean, x='Season', y='Disease_Pest_Risk_pct', hue='Season', order=['Kharif', 'Rabi', 'Zaid'], palette='Reds', ax=ax, legend=False)
ax.set_title('Disease/Pest Risk (%) by Season')
plt.tight_layout()
plt.show()

print(df_clean.groupby('Season')['Disease_Pest_Risk_pct'].describe().round(2))


**Observation:** Kharif season's higher humidity and rainfall are generally associated with greater disease/pest pressure in real-world agronomy — check whether that pattern holds here, and note it as a seasonal risk factor for planning.


## 12. Regional Patterns Across Seasons

**Key question:** Are seasonal patterns consistent across different states, or do some regions behave differently?


In [ ]:
state_season_yield = df_clean.pivot_table(values='Yield_Tonnes_Ha', index='State', columns='Season', aggfunc='mean').round(2)
state_season_yield


In [ ]:
state_season_profit = df_clean.pivot_table(values='Profit_INR', index='State', columns='Season', aggfunc='mean').round(0)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(state_season_profit, annot=True, fmt='.0f', cmap='RdYlGn', center=0, ax=ax, cbar_kws={'label': 'Avg Profit (INR)'})
ax.set_title('Average Profit by State and Season')
plt.tight_layout()
plt.show()


## 13. Relationships Between Seasonal Conditions and Outcomes

**Key question:** Are there relationships between environmental conditions and agricultural performance?


In [ ]:
numeric_cols = ['Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct', 'Sunlight_Hours_Day',
                'Soil_pH', 'Soil_Moisture_pct', 'Nitrogen_kg_ha', 'Phosphorus_kg_ha',
                'Potassium_kg_ha', 'Fertilizer_kg_ha', 'Pesticide_Litre_ha', 'Seed_Quality_Score',
                'Yield_Tonnes_Ha', 'Profit_INR', 'Water_Efficiency_t_per_1000m3', 'Disease_Pest_Risk_pct']

corr = df_clean[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Matrix: Environmental & Resource Factors vs Outcomes')
plt.tight_layout()
plt.show()


In [ ]:
# Correlation of each factor specifically with Yield and Profit, sorted
print("Correlation with Yield_Tonnes_Ha:")
print(corr['Yield_Tonnes_Ha'].sort_values(ascending=False))
print("\nCorrelation with Profit_INR:")
print(corr['Profit_INR'].sort_values(ascending=False))


**Observation:** Interpret the strongest positive and negative correlations here — e.g. which environmental or input factors move most closely with yield and profit — and connect them back to seasonal patterns observed above.


## 14. Statistical Test: Is the Seasonal Difference in Yield Significant?

A one-way ANOVA tests whether the mean normalized yield differs significantly across the three seasons.


In [ ]:
from scipy import stats

groups = [norm[norm['Season'] == s]['Yield_Normalized'].dropna() for s in ['Kharif', 'Rabi', 'Zaid']]
f_stat, p_value = stats.f_oneway(*groups)

print(f"F-statistic: {f_stat:.3f}")
print(f"P-value: {p_value:.5f}")

if p_value < 0.05:
    print("\nResult: Statistically significant difference in normalized yield across seasons (p < 0.05).")
else:
    print("\nResult: No statistically significant difference in normalized yield across seasons (p >= 0.05).")
